In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/mac/Studying/OTTO/scenario_generator


In [3]:
# Frame Scheduler demonstration
from collections import Counter
from src.config_loader import ConfigLoader
from src.signal_generator import Scenario, SignalGenerator
from src.can_encoder import CANEncoder
from src.frame_scheduler import FrameScheduler

loader = ConfigLoader("../config")
gen = SignalGenerator(loader)
vehicles = loader.load_vehicles()
signals_cfg = loader.load_signals()

# --- Generate + encode ---
scenario = Scenario.from_yaml("../config/scenarios/city_drive.yaml")
df = gen.generate(scenario)

encoder = CANEncoder(vehicles["sedan_a"], signals_cfg)
frames_by_id = {}
for frames in encoder.encode_dataframe(df):
    for can_id, payload in frames.items():
        frames_by_id.setdefault(can_id, []).append(payload)

# --- Schedule ---
messages = loader.get_messages_for_vehicle("sedan_a")
scheduler = FrameScheduler(messages, base_rate_hz=100, duration_s=60.0)
scheduled = scheduler.schedule(frames_by_id)

# --- Summary ---
print(f"Total frames scheduled: {len(scheduled)}")
print(f"Expected: {scheduler.expected_frame_count()}")
print()

per_message = Counter(f.can_id for f in scheduled)
print("Frames per CAN ID:")
for can_id, count in sorted(per_message.items()):
    cycle_ms = next(m["cycle_ms"] for m in messages.values() if m["id"] == can_id)
    print(f"  0x{can_id:03X}  cycle={cycle_ms:>3} ms  frames={count}")

print()
print("First 20 frames (chronological):")
print(f"{'t_ms':>8}  {'can_id':>8}  payload")
for f in scheduled[:20]:
    print(f"{f.timestamp_ms:>8}  0x{f.can_id:03X}    {f.payload.hex()}")

print()
print("Frames around t=1000 ms (1 s):")
window = [f for f in scheduled if 990 <= f.timestamp_ms <= 1010]
for f in window:
    print(f"{f.timestamp_ms:>8}  0x{f.can_id:03X}    {f.payload.hex()}")

Total frames scheduled: 10200
Expected: 10200

Frames per CAN ID:
  0x100  cycle= 10 ms  frames=6000
  0x120  cycle= 50 ms  frames=1200
  0x1A0  cycle= 20 ms  frames=3000

First 20 frames (chronological):
    t_ms    can_id  payload
       0  0x100    0c80003c00000000
       0  0x120    000003e800000000
       0  0x1A0    0000000000000000
      10  0x100    0c80003c00000000
      20  0x100    0c80003c00000000
      20  0x1A0    0000000000000000
      30  0x100    0c80003c00000000
      40  0x100    0c80003c00000000
      40  0x1A0    0000000000000000
      50  0x100    0c80003c00000000
      50  0x120    000003e800000000
      60  0x100    0c80003c00000000
      60  0x1A0    0000000000000000
      70  0x100    0c80003c00000000
      80  0x100    0c80003c00000000
      80  0x1A0    0000000000000000
      90  0x100    0c80003c00000000
     100  0x100    0c80003c00000000
     100  0x120    000003e800000000
     100  0x1A0    0000000000000000

Frames around t=1000 ms (1 s):
     990  0x100

In [4]:
#  Inspect frames around the middle of the scenario
from src.config_loader import ConfigLoader
from src.signal_generator import Scenario, SignalGenerator
from src.can_encoder import CANEncoder
from src.frame_scheduler import FrameScheduler


loader = ConfigLoader("../config")
gen = SignalGenerator(loader)
vehicles = loader.load_vehicles()
signals_cfg = loader.load_signals()

scenario = Scenario.from_yaml("../config/scenarios/city_drive.yaml")
df = gen.generate(scenario)

encoder = CANEncoder(vehicles["sedan_a"], signals_cfg)
frames_by_id = {}
for frames in encoder.encode_dataframe(df):
    for can_id, payload in frames.items():
        frames_by_id.setdefault(can_id, []).append(payload)

messages = loader.get_messages_for_vehicle("sedan_a")
scheduler = FrameScheduler(messages, base_rate_hz=100, duration_s=60.0)
scheduled = scheduler.schedule(frames_by_id)

# --- Compute the middle timestamp ---
midpoint_ms = int(scenario.duration_s * 1000 / 2)  # 30000 ms for 60 s
print(f"Scenario: {scenario.name}  |  Duration: {scenario.duration_s} s")
print(f"Midpoint: {midpoint_ms} ms ({midpoint_ms / 1000:.1f} s)")
print()

# --- Show a window around the midpoint ---
window_ms = 100  # ±100 ms around the midpoint
window = [
    f for f in scheduled
    if abs(f.timestamp_ms - midpoint_ms) <= window_ms
]

print(f"Frames within ±{window_ms} ms of t={midpoint_ms} ms:")
print(f"{'t_ms':>8}  {'can_id':>8}  payload           decoded_signals")
print("-" * 80)

for f in window:
    # Decode payload into readable signal values
    decoded = encoder.decode(f.can_id, f.payload)
    decoded_str = "  ".join(f"{k}={v:.2f}" for k, v in decoded.items())
    print(f"{f.timestamp_ms:>8}  0x{f.can_id:03X}    {f.payload.hex()}  {decoded_str}")

# --- Also show what the raw signals were at exactly t=30.0 s ---
print()
print(f"--- Raw signal values at t = {midpoint_ms / 1000:.1f} s ---")
mid_idx = midpoint_ms // 10  # 10 ms base step
row = df.iloc[mid_idx]
for sig in encoder.mapped_signal_names():
    print(f"  {sig:30s}  {row[sig]:10.3f}")

Scenario: City Drive  |  Duration: 60.0 s
Midpoint: 30000 ms (30.0 s)

Frames within ±100 ms of t=30000 ms:
    t_ms    can_id  payload           decoded_signals
--------------------------------------------------------------------------------
   29900  0x100    0c80004300000000  engine_rpm=800.00  throttle_position=0.00  coolant_temperature=27.00
   29900  0x120    000003e800000000  vehicle_speed=0.00  longitudinal_acceleration=0.00
   29900  0x1A0    0000000000000000  brake_pressure=0.00  brake_pedal=0.00  wheel_speed=0.00
   29910  0x100    0c80004300000000  engine_rpm=800.00  throttle_position=0.00  coolant_temperature=27.00
   29920  0x100    0c80004300000000  engine_rpm=800.00  throttle_position=0.00  coolant_temperature=27.00
   29920  0x1A0    0000000000000000  brake_pressure=0.00  brake_pedal=0.00  wheel_speed=0.00
   29930  0x100    0c80004300000000  engine_rpm=800.00  throttle_position=0.00  coolant_temperature=27.00
   29940  0x100    0c80004300000000  engine_rpm=800.00  thr